# GTO integral tests

In the previous notebook, we defined the molecular integrals $S$, $T$, $V$, and $(ab|cd)$ in terms of the primitive integrals of 3D gaussians. 

In this notebook, we will compare the implementation of the primitive integral evaluations with known reference values. For this we will use PySCF as reference. We will define a Helium and a hydrogen atoms separated by 1 in the $x$ axis: 

In [23]:
import numpy as np
from pyscf import gto

from py_mods.src.integrals.GTO import S_3D, T_3D, V_3D, eri
from py_mods.src.integrals.GTO import GTO, create_normalized_GTO

First we try the $1s/1s$:

In [24]:
basis_dict = {"H": [[0, [0.5, 1.0]]], "He": [[0, [0.4, 1.0]]]}
He_H_mol = gto.M(
    atom="He 0 0 0; H 1 0 0", basis=basis_dict, unit="Bohr", spin=1, verbose=0
)

He_1s = create_normalized_GTO(np.array([0, 0, 0]), 0.4, 0)
H_1s = create_normalized_GTO(np.array([1, 0, 0]), 0.5, 0)

And we define the same as:

In [25]:
print(repr(He_1s))

GTO(R=array([0, 0, 0]), exp=0.4, total_L=0, l_projections=array([[0, 0, 0]], dtype=int32), normalization_constants=array([0.35847187]), charge=1)


In [26]:
S_1s1s = He_H_mol.intor_symmetric("int1e_ovlp")
print(f"S_1s1s (PySCF): {S_1s1s[0,1]:.10f}")

S_1s1s_self = S_3D(
    He_1s,
    He_1s.l_projections[0],
    He_1s.normalization_constants[0],
    H_1s,
    H_1s.l_projections[0],
    H_1s.normalization_constants[0],
)
print(f"S_1s1s (Oatmeal): {S_1s1s_self:.10f}")

S_1s1s (PySCF): 0.7933116667
S_1s1s (Oatmeal): 0.7933116667


In [27]:
T_1s1s = He_H_mol.intor_symmetric("int1e_kin")
print(f"T_1s1s (PySCF): {T_1s1s[0,1]:.10f}")

T_1s1s_self = T_3D(
    He_1s,
    He_1s.l_projections[0],
    He_1s.normalization_constants[0],
    H_1s,
    H_1s.l_projections[0],
    H_1s.normalization_constants[0],
)
print(f"T_1s1s (Oatmeal): {T_1s1s_self:.10f}")

T_1s1s (PySCF): 0.4505226749
T_1s1s (Oatmeal): 0.4505226749


In [28]:
V_1s1s = He_H_mol.intor_symmetric("int1e_nuc")
print(f"V_1s1s (PySCF): {V_1s1s[0,1]:.10f}")

V_He = V_3D(
    He_1s,
    He_1s.l_projections[0],
    He_1s.normalization_constants[0],
    H_1s,
    H_1s.l_projections[0],
    H_1s.normalization_constants[0],
    2.0,
    np.array([0.0, 0.0, 0.0]),
) + V_3D(
    He_1s,
    He_1s.l_projections[0],
    He_1s.normalization_constants[0],
    H_1s,
    H_1s.l_projections[0],
    H_1s.normalization_constants[0],
    1.0,
    np.array([1.0, 0.0, 0.0]),
)



print(f"V_1s_He (Oatmeal): {V_He}")



V_1s1s (PySCF): -2.3549300013
V_1s_He (Oatmeal): -2.354930001277591


In [29]:
eri_1s1s_pyscf = He_H_mol.intor("int2e")

In [30]:
self_eris = np.zeros([2, 2, 2, 2])

for i, prim_a in enumerate([He_1s, H_1s]):
    for j, prim_b in enumerate([He_1s, H_1s]):
        for k, prim_c in enumerate([He_1s, H_1s]):
            for l, prim_d in enumerate([He_1s, H_1s]):
                self_eris[i, j, k, l] = eri(
                    prim_a,
                    prim_a.l_projections[0],
                    prim_a.normalization_constants[0],
                    prim_b,
                    prim_b.l_projections[0],
                    prim_b.normalization_constants[0],
                    prim_c,
                    prim_c.l_projections[0],
                    prim_c.normalization_constants[0],
                    prim_d,
                    prim_d.l_projections[0],
                    prim_d.normalization_constants[0],
                )

In [31]:
np.all(np.abs(self_eris - eri_1s1s_pyscf) < 1e-7)

np.True_

And we can see that the current implementation works for S primitives!!

---

# Integral test for $p$ functions
Now we will test for $p-p$ integrals

In [32]:
basis_dict = {"H": [[1, [0.5, 1.0]]], "He": [[1, [0.4, 1.0]]]}
He_H_2p_mol = gto.M(
    atom="He 0 0 0; H 1 0 0", basis=basis_dict, unit="Bohr", spin=1, verbose=0
)

He_2p = create_normalized_GTO(np.array([0, 0, 0]), 0.4, 1)
H_2p = create_normalized_GTO(np.array([1, 0, 0]), 0.5, 1)

In [33]:
print(repr(He_2p))

GTO(R=array([0, 0, 0]), exp=0.4, total_L=1, l_projections=array([[1, 0, 0],
       [0, 1, 0],
       [0, 0, 1]], dtype=int32), normalization_constants=array([0.45343504, 0.45343504, 0.45343504]), charge=1)


In [34]:
S_2p2p = He_H_2p_mol.intor_symmetric("int1e_ovlp")
print(f"S_2p2p (PySCF):\n {repr(S_2p2p)}")

S_2p2p (PySCF):
 array([[ 1.        ,  0.        ,  0.        ,  0.43799971,  0.        ,
         0.        ],
       [ 0.        ,  1.        ,  0.        , -0.        ,  0.78839947,
         0.        ],
       [ 0.        ,  0.        ,  1.        , -0.        ,  0.        ,
         0.78839947],
       [ 0.43799971, -0.        , -0.        ,  1.        ,  0.        ,
         0.        ],
       [ 0.        ,  0.78839947,  0.        ,  0.        ,  1.        ,
         0.        ],
       [ 0.        ,  0.        ,  0.78839947,  0.        ,  0.        ,
         1.        ]])


In [35]:
S_2p2p_self = np.zeros([2 * 3, 2 * 3])

for i, prim_a in enumerate([He_2p, H_2p]):
    for j, prim_b in enumerate([He_2p, H_2p]):
        for la in range(3):
            for lb in range(3):
                S_2p2p_self[i * 3 + la, j * 3 + lb] = S_3D(
                    prim_a,
                    prim_a.l_projections[la],
                    prim_a.normalization_constants[0],
                    prim_b,
                    prim_b.l_projections[lb],
                    prim_b.normalization_constants[0],
                )

print(f"S_2p2p (Oatmeal):\n {S_2p2p_self}")

S_2p2p (Oatmeal):
 [[ 1.          0.          0.          0.43799971  0.          0.        ]
 [ 0.          1.          0.         -0.          0.78839947  0.        ]
 [ 0.          0.          1.         -0.          0.          0.78839947]
 [ 0.43799971 -0.         -0.          1.          0.          0.        ]
 [ 0.          0.78839947  0.          0.          1.          0.        ]
 [ 0.          0.          0.78839947  0.          0.          1.        ]]


In [36]:
print(np.all(np.abs(S_2p2p - S_2p2p_self) < 1e-7))

True


In [37]:
T_2p2p = He_H_2p_mol.intor_symmetric("int1e_kin")
print(f"T_2p2p (PySCF):\n {repr(T_2p2p)}")

T_2p2p (PySCF):
 array([[ 1.        ,  0.        ,  0.        ,  0.28767388,  0.        ,
         0.        ],
       [ 0.        ,  1.        ,  0.        , -0.        ,  0.7981328 ,
         0.        ],
       [ 0.        ,  0.        ,  1.        , -0.        ,  0.        ,
         0.7981328 ],
       [ 0.28767388, -0.        , -0.        ,  1.25      ,  0.        ,
         0.        ],
       [ 0.        ,  0.7981328 ,  0.        ,  0.        ,  1.25      ,
         0.        ],
       [ 0.        ,  0.        ,  0.7981328 ,  0.        ,  0.        ,
         1.25      ]])


In [38]:
T_2p2p_self = np.zeros([2 * 3, 2 * 3])
for i, prim_a in enumerate([He_2p, H_2p]):
    for j, prim_b in enumerate([He_2p, H_2p]):
        for la in range(3):
            for lb in range(3):
                T_2p2p_self[i * 3 + la, j * 3 + lb] = T_3D(
                    prim_a,
                    prim_a.l_projections[la],
                    prim_a.normalization_constants[0],
                    prim_b,
                    prim_b.l_projections[lb],
                    prim_b.normalization_constants[0],
                )

print(f"T_2p2p (Oatmeal):\n {T_2p2p_self}")

T_2p2p (Oatmeal):
 [[ 1.          0.          0.          0.28767388  0.          0.        ]
 [ 0.          1.          0.         -0.          0.7981328   0.        ]
 [ 0.          0.          1.         -0.          0.          0.7981328 ]
 [ 0.28767388 -0.         -0.          1.25        0.          0.        ]
 [ 0.          0.7981328   0.          0.          1.25        0.        ]
 [ 0.          0.          0.7981328   0.          0.          1.25      ]]


In [39]:
print(np.all(np.abs(T_2p2p - T_2p2p_self) < 1e-7))

True


In [40]:
V_2p2p_pyscf = He_H_2p_mol.intor_symmetric("int1e_nuc")


In [41]:
V_2p2p_self = np.zeros([2 * 3, 2 * 3])
for i, prim_a in enumerate([He_2p, H_2p]):
    for j, prim_b in enumerate([He_2p, H_2p]):
        for la in range(3):
            for lb in range(3):
                V_He = V_3D(
                    prim_a,
                    prim_a.l_projections[la],
                    prim_a.normalization_constants[la],
                    prim_b,
                    prim_b.l_projections[lb],
                    prim_b.normalization_constants[lb],
                    2.0,
                    np.array([0.0, 0.0, 0.0]),
                )

                V_H = V_3D(
                    prim_a,
                    prim_a.l_projections[la],
                    prim_a.normalization_constants[la],
                    prim_b,
                    prim_b.l_projections[lb],
                    prim_b.normalization_constants[lb],
                    1.0,
                    np.array([1.0, 0.0, 0.0]),
                )
                V_2p2p_self[i * 3 + la, j * 3 + lb] = V_He + V_H

print(f"V_2p2p (PySCF):\n {repr(V_2p2p_pyscf)}")

print(f"V_2p2p (Oatmeal):\n {V_2p2p_self}")

V_2p2p (PySCF):
 array([[-2.11204358,  0.        ,  0.        , -0.74592823,  0.        ,
         0.        ],
       [ 0.        , -1.9268861 ,  0.        ,  0.        , -1.60967793,
         0.        ],
       [ 0.        ,  0.        , -1.9268861 ,  0.        ,  0.        ,
        -1.60967793],
       [-0.74592823,  0.        ,  0.        , -2.46262596,  0.        ,
         0.        ],
       [ 0.        , -1.60967793,  0.        ,  0.        , -2.01006107,
         0.        ],
       [ 0.        ,  0.        , -1.60967793,  0.        ,  0.        ,
        -2.01006107]])
V_2p2p (Oatmeal):
 [[-2.11204358 -0.         -0.         -0.74592823 -0.         -0.        ]
 [-0.         -1.9268861  -0.         -0.         -1.60967793 -0.        ]
 [-0.         -0.         -1.9268861  -0.         -0.         -1.60967793]
 [-0.74592823 -0.         -0.         -2.46262596 -0.         -0.        ]
 [-0.         -1.60967793 -0.         -0.         -2.01006107 -0.        ]
 [-0.         -0. 

In [42]:
eri_2p2p2p2p_pyscf = He_H_2p_mol.intor("int2e")
eri_2p2p2p2p_self = np.zeros([2 * 3, 2 * 3, 2 * 3, 2 * 3])

for i, prim_a in enumerate([He_2p, H_2p]):
    for j, prim_b in enumerate([He_2p, H_2p]):
        for k, prim_c in enumerate([He_2p, H_2p]):
            for l, prim_d in enumerate([He_2p, H_2p]):
                for la in range(3):
                    for lb in range(3):
                        for lc in range(3):
                            for ld in range(3):
                                eri_2p2p2p2p_self[
                                    i * 3 + la, j * 3 + lb, k * 3 + lc, l * 3 + ld
                                ] = eri(
                                    prim_a,
                                    prim_a.l_projections[la],
                                    prim_a.normalization_constants[la],
                                    prim_b,
                                    prim_b.l_projections[lb],
                                    prim_b.normalization_constants[lb],
                                    prim_c,
                                    prim_c.l_projections[lc],
                                    prim_c.normalization_constants[lc],
                                    prim_d,
                                    prim_d.l_projections[ld],
                                    prim_d.normalization_constants[ld],
                                )



In [43]:
for i in range(2 * 3):
    for j in range(2 * 3):
        if not np.all(
            np.abs(eri_2p2p2p2p_self[i, j, :, :] - eri_2p2p2p2p_pyscf[i, j, :, :])
            < 1e-7
        ):
            print("Incorrect eri calculated with indices: ", i, j)

In [44]:
print(np.all(np.abs(S_2p2p - S_2p2p_self) < 1e-7))

True


And at this moment $1s$ and $2p$ seem to work. 

----

However, it is clear we need a cleaner interface.